In [1]:
import os
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader,Subset 
from PIL import Image
import kagglehub

#dataset download

In [2]:
import kagglehub

path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path:", path)

path: C:\Users\alomg\.cache\kagglehub\datasets\mohitsingh1804\plantvillage\versions\1


In [3]:
#setup
torch.manual_seed(42)

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device:{device}')

Using device:cpu


In [37]:
#path

TRAIN_PATH=os.path.join(path,"plantvillage","train")
VAL_PATH = os.path.join(path,"plantvillage","val")



In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3,std=[0.5]*3)
    ]
    
)

In [6]:
#Custom Dataset Class

In [7]:
class MultiClassClassfication(Dataset):
    def __init__(self,root_dir,transform=None):
        super().__init__()

        self.samples=[]
        self.transfrom= transform

        self.classes=sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])

        #idx the classes
        self.class_to_idx= {cls_name : idx for idx,cls_name in enumerate(self.classes)}

        for class_name in self.classes:
            class_path=os.path.join(root_dir,class_name)
            for img_name in os.listdir(class_path):
                img_path=os.path.join(class_path,img_name)
                if os.path.isfile(img_path):
                    label= self.class_to_idx[class_name]
                    self.samples.append((img_path,label))
            
    def __len__(self):
        return len(self.samples)

    def __getitem__(self,index):
        img_path,label=self.samples[index]
        image=Image.open(img_path).convert("RGB")
        if self.transfrom:
            image=self.transfrom(image)
        return image,label

        

In [8]:
#load data

In [9]:
train_dataset_full=MultiClassClassfication(TRAIN_PATH,transform)
test_dataset_full=MultiClassClassfication(VAL_PATH,transform)
num_classes=len(train_dataset_full.classes)

print("Number of Classes: ",num_classes)
print("Full Train size :",len(train_dataset_full)) 
print("Full Test size :",len(test_dataset_full))

Number of Classes:  38
Full Train size : 43444
Full Test size : 10861


# Use a subset for fast testing (optional)

In [21]:
train_dataset=Subset(train_dataset_full,list(range(min(1000,len(train_dataset_full)))))
test_dataset=Subset(test_dataset_full,list(range(min(1000,len(train_dataset_full)))))

In [22]:
#Data Loader

In [23]:
pin= True if device.type =='cuda' else False
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory_device=pin)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory_device=pin)

In [24]:
#Nural Links

In [25]:
class MYCNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32), 
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64), 
            nn.MaxPool2d(2),
            nn.Conv2d(64,128,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128), 
            nn.MaxPool2d(2),
        )
        self.classifiers=nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16,128),
            nn.ReLU(),
            nn.Dropout(0.4), 
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64,num_classes)
        )
    def forward(self,X):
        x=self.features(X)
        x=self.classifiers(x) 
        return x

In [26]:
model=MYCNN(num_classes=num_classes).to(device)

In [27]:
#Training Setup

In [28]:
learning_rate=0.001 
epochs=10
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=learning_rate)


In [29]:
#training Loop

In [30]:
for epoch in range(epochs): 
    model.train() 
    total_loss=0
    for batch_features,batch_labels in train_loader: 
        batch_features=batch_features.to(device)
        batch_labels=batch_labels.to(device)
        outputs=model(batch_features)
        loss=criterion(outputs,batch_labels)
        optimizer.zero_grad()
        loss.backward() 
        total_loss+=loss.item() 
    avg_loss= total_loss/len(train_loader)
    print(f"epoch {epoch+1}/{epochs} ,Loss: {avg_loss:.4f}")

epoch 1/10 ,Loss: 3.7108
epoch 2/10 ,Loss: 3.7086
epoch 3/10 ,Loss: 3.7013
epoch 4/10 ,Loss: 3.7036
epoch 5/10 ,Loss: 3.7027
epoch 6/10 ,Loss: 3.7133
epoch 7/10 ,Loss: 3.7059
epoch 8/10 ,Loss: 3.7060
epoch 9/10 ,Loss: 3.7060
epoch 10/10 ,Loss: 3.7037


In [31]:
#Evualtion Function

In [34]:
def evaluate(loader): 
    model.eval() 
    total,correct=0,0 
    with torch.no_grad(): 
        for batch_features,batch_labels in loader: 
            batch_features,batch_labels=batch_features.to(device),batch_labels.to(device) 
            outputs=model(batch_features)
            _,predicted= torch.max(outputs,1)
            total+=batch_labels.size(0)
            correct+= (predicted== batch_labels).sum().item() 
    return correct/total 

In [35]:
test_acc= evaluate(test_loader)
print("Test accuracy: ",test_acc)

Test accuracy:  0.003
